<a href="https://colab.research.google.com/github/yeaminshakhor/bengali-cyberbullying-detection-data-quality/blob/main/DataAnalyzingCSV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets torch scikit-learn statsmodels


In [ ]:
import pandas as pd, numpy as np, torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
)
from torch.utils.data import Dataset

MODEL_NAME = "sagorsarker/bangla-bert-base"

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(list(texts), truncation=True,
                                   padding=True, max_length=max_len)
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

# -------- load --------
synth   = pd.read_csv("canonical_training_dataset_v2.csv", encoding="utf-8-sig")
holdout = pd.read_csv("organic_holdout_test_set.csv",     encoding="utf-8-sig")

train_df = synth[["text", "label"]].dropna().drop_duplicates().reset_index(drop=True)
test_df  = holdout[["text", "label"]].dropna().reset_index(drop=True)

print(f"Train rows : {len(train_df)}")
print(f"Test  rows : {len(test_df)}")
print("Train dist :", train_df["label"].value_counts().to_dict())
print("Test  dist :", test_df["label"].value_counts().to_dict())

# -------- model + tokenizer --------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

train_ds = TextDataset(train_df["text"], train_df["label"], tokenizer)
test_ds  = TextDataset(test_df["text"],  test_df["label"],  tokenizer)

# -------- training --------
args = TrainingArguments(
    output_dir="./banglabert_out",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print("\nFinal evaluation on holdout:")
print(trainer.evaluate())

# Save the fine-tuned model
model.save_pretrained("./banglabert_cyberbullying_final")
tokenizer.save_pretrained("./banglabert_cyberbullying_final")
print("\nSaved model to ./banglabert_cyberbullying_final")

Train rows : 4713
Test  rows : 1172
Train dist : {1: 2473, 0: 2240}
Test  dist : {1: 932, 0: 240}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


In [ ]:
from google.colab import files
files.download("banglabert_preds_on_holdout.csv")

In [ ]:
import pandas as pd
import joblib
import os
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from statsmodels.stats.contingency_tables import mcnemar

# Verify if our newly generated predictions exist
bert_pred_path = "preds_on_organic_holdout_test_set.csv"
if not os.path.exists(bert_pred_path):
    # Fall back to default name if the loop has not run yet
    bert_pred_path = "banglabert_preds_on_holdout.csv"

# Load original holdout evaluation dataset
holdout = pd.read_csv("organic_holdout_test_set.csv", encoding="utf-8-sig")
y_true  = holdout["label"].values

# Load RandomForest baseline
vec  = joblib.load("vectorizer_combined.pkl")
rf   = joblib.load("bullying_model_combined.pkl")
y_rf = rf.predict(vec.transform(holdout["text"].astype(str)))

# Load Fine-tuned BanglaBERT Predictions
bert   = pd.read_csv(bert_pred_path, encoding="utf-8-sig")
y_bert = bert["pred"].values

assert len(y_true) == len(y_rf) == len(y_bert), \
    f"Length mismatch: {len(y_true)}, {len(y_rf)}, {len(y_bert)}"
assert (holdout["text"].astype(str).values ==
        bert["text"].astype(str).values).all(), \
    "Row order mismatch between holdout and BanglaBERT CSV"

def summarize(name, y):
    return {
        "model":      name,
        "accuracy":   accuracy_score(y_true, y),
        "macro_f1":   f1_score(y_true, y, average="macro"),
        "prec_bully": precision_score(y_true, y, pos_label=1, zero_division=0),
        "rec_bully":  recall_score(y_true, y, pos_label=1, zero_division=0),
    }

summary = pd.DataFrame([summarize("RF (C)", y_rf),
                        summarize("BERT (D)", y_bert)])
print(f"\n=== Metrics on organic holdout (n={len(y_true)}) ===")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nRF confusion [ [TN FP],[FN TP] ]:")
print(confusion_matrix(y_true, y_rf))
print("BERT confusion [ [TN FP],[FN TP] ]:")
print(confusion_matrix(y_true, y_bert))

a = int(((y_rf == y_true) & (y_bert == y_true)).sum())
b = int(((y_rf == y_true) & (y_bert != y_true)).sum())
c = int(((y_rf != y_true) & (y_bert == y_true)).sum())
d = int(((y_rf != y_true) & (y_bert != y_true)).sum())

res = mcnemar([[a, b], [c, d]], exact=False, correction=True)

print("\n=== McNemar (RF vs BERT) ===")
print(f"table [[a,b],[c,d]] = [[{a},{b}],[{c},{d}]]")
print(f"chi2 = {res.statistic:.4f}")
print(f"p    = {res.pvalue:.6g}")
print(f"discordant: RF-only-correct={b}, BERT-only-correct={c}")

if res.pvalue < 0.05:
    print("\n>> BanglaBERT significantly outperforms RandomForest (p < 0.05)")
else:
    print("\n>> No significant difference (p >= 0.05) — report as a trend")

In [ ]:
from google.colab import files
import glob

# Let's download the generated predictions for the holdout set
predictions_file = "preds_on_organic_holdout_test_set.csv"
import os
if os.path.exists(predictions_file):
    print(f"Downloading {predictions_file}...")
    files.download(predictions_file)
else:
    print("Predictions file not found. Make sure to run cell 14560bf3 first.")

In [ ]:
from google.colab import files
import glob
import os

# Find all generated prediction files
pred_files = glob.glob("/content/preds_on_*.csv")

print(f"Found {len(pred_files)} prediction files ready to download.")
for f in pred_files:
    filename = os.path.basename(f)
    print(f"Downloading: {filename}")
    files.download(f)

In [ ]:
import glob
import os

# Find all CSV files in the environment
all_csv_files = glob.glob("/content/*.csv")
print(f"Found {len(all_csv_files)} CSV files:")
for f in all_csv_files:
    print(f" - {os.path.basename(f)}")

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

# Loading your newly trained BanglaBERT cyberbullying classifier
try:
    tokenizer = AutoTokenizer.from_pretrained("./banglabert_cyberbullying_final")
    model = AutoModelForSequenceClassification.from_pretrained("./banglabert_cyberbullying_final")
    print("Successfully loaded the custom fine-tuned BanglaBERT model!")
except Exception as e:
    print(f"Could not load fine-tuned model: {e}. Falling back to base model.")
    tokenizer = AutoTokenizer.from_pretrained("sagorsarker/bangla-bert-base")
    model = AutoModelForSequenceClassification.from_pretrained("sagorsarker/bangla-bert-base", num_labels=2)

trainer = Trainer(model=model)

In [ ]:
class PredictionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
    def __len__(self):
        return len(self.encodings["input_ids"])

# Run prediction loop across all detected CSV files
for csv_path in all_csv_files:
    file_name = os.path.basename(csv_path)
    if file_name in ["canonical_training_dataset_v2.csv", "canonical_training_dataset.csv"]:
        continue # Skip training sources

    print(f"\nProcessing predictions for: {file_name}")
    try:
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
        if "text" not in df.columns:
            print(f"Skipping {file_name} because 'text' column is missing.")
            continue

        # Prepare inputs
        texts = df["text"].astype(str).fillna("").values
        pred_ds = PredictionDataset(texts, tokenizer)

        # Predict
        logits = trainer.predict(pred_ds).predictions
        preds = logits.argmax(-1)
        probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]

        df["pred"] = preds
        df["prob"] = probs

        out_name = f"preds_on_{file_name}"
        df.to_csv(out_name, index=False, encoding="utf-8-sig")
        print(f"Saved {len(df)} predictions to {out_name}")
    except Exception as e:
        print(f"Error processing {file_name}: {e}")